# Aula 6 — (Opcional) Fine-tuning com LoRA no Google Colab

**Fora da nota.** Ajusta um modelo aberto pequeno com o dataset do "tom da Vovó" usando LoRA.

Como usar:
1. Abra este arquivo no Google Colab (File → Upload notebook).
2. Menu **Ambiente de execução → Alterar tipo → GPU T4** (gratuita).
3. Rode as células em ordem. Leva cerca de 10 a 15 minutos.

O que observar: a resposta do modelo **antes** e **depois** do ajuste.
Este notebook não usa a OpenAI nem a sua chave.

In [ ]:
!pip install -q "transformers>=4.53" "peft>=0.15" "trl>=0.19" "datasets>=3.0" accelerate

## 1. Dataset
Troque `SEU_USUARIO` pelo seu usuário do GitHub (o fork precisa ser público),
ou faça upload do arquivo `tom_vovo.jsonl` pelo painel lateral do Colab e use o caminho local.

In [ ]:
from datasets import load_dataset

URL = "https://raw.githubusercontent.com/SEU_USUARIO/engenharia-ia-fia/main/dados/finetune/tom_vovo.jsonl"
dataset = load_dataset("json", data_files=URL, split="train")   # ou data_files="tom_vovo.jsonl"
print(dataset)
print(dataset[0]["messages"])

## 2. Modelo base e resposta ANTES do ajuste

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO = "Qwen/Qwen2.5-0.5B-Instruct"   # pequeno o bastante para a T4 gratuita
tokenizer = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.float16, device_map="auto")

def perguntar(m, pergunta):
    mensagens = [
        {"role": "system", "content": dataset[0]["messages"][0]["content"]},
        {"role": "user", "content": pergunta},
    ]
    entrada = tokenizer.apply_chat_template(mensagens, add_generation_prompt=True, return_tensors="pt").to(m.device)
    saida = m.generate(entrada, max_new_tokens=80, do_sample=False)
    return tokenizer.decode(saida[0][entrada.shape[1]:], skip_special_tokens=True)

print("ANTES:", perguntar(modelo, "Vocês têm café?"))

## 3. Treino com LoRA
LoRA congela o modelo original e treina só pequenas matrizes "adaptadoras" (menos de 1% dos parâmetros).

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
config = SFTConfig(
    output_dir="vovo-lora",
    num_train_epochs=8,              # dataset pequeno: mais épocas
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=True,
)
treinador = SFTTrainer(model=modelo, args=config, train_dataset=dataset, peft_config=lora, processing_class=tokenizer)
treinador.train()

## 4. DEPOIS do ajuste

In [ ]:
modelo_ajustado = treinador.model
for pergunta in ["Vocês têm café?", "Que horas fecha no sábado?", "O sonho é bom?"]:
    print(f"P: {pergunta}\nR: {perguntar(modelo_ajustado, pergunta)}\n")

## 5. Para discutir
- O tom mudou? E os **fatos** (preços, horários), continuam certos? Por que o RAG ainda é necessário?
- Com 12 exemplos, o modelo generaliza ou decora?
- Quanto custou (tempo de GPU) e quanto custaria manter isso em produção, comparado ao few-shot?
- O adaptador LoRA salvo tem poucos megabytes: `treinador.model.save_pretrained("vovo-lora")`.